# Processing Logs with EnkryptAI Guardrails

This notebook demonstrates how to efficiently process large volumes of log data through EnkryptAI Guardrails to detect policy violations, security issues, and compliance problems.

## What You'll Learn

- How to load and process log files containing user prompts
- How to batch process logs efficiently using EnkryptAI's batch API
- How to handle rate limiting with exponential backoff
- How to analyze and save guardrails results

## Use Cases

- **Security Audits**: Review historical logs for security violations
- **Compliance Checks**: Ensure all user interactions meet policy requirements
- **Quality Assurance**: Identify problematic patterns in user inputs
- **Incident Investigation**: Analyze logs around specific time periods or events


## Step 1: Setup and Configuration

First, we'll import the necessary libraries and configure our environment. Make sure you have:
- Your EnkryptAI API key set in a `.env` file
- A `logs.json` file with your log data (each log should have a `prompt` field)


In [ ]:
import requests
import time
from dotenv import load_dotenv
import os
import json

# Load environment variables from .env file
load_dotenv()

# Configuration
ENKRYPTAI_API_KEY = os.getenv("ENKRYPTAI_API_KEY")
ENKRYPTAI_GUARDRAILS_NAME = os.getenv("ENKRYPTAI_GUARDRAILS_NAME")  # Your guardrail policy name
LOGS_FILE_PATH = "logs.json"  # Path to your logs file
BATCH_SIZE = 10  # Number of logs to process in each batch

# Validate configuration
if not ENKRYPTAI_API_KEY:
    raise ValueError("ENKRYPTAI_API_KEY not found in environment variables. Please set it in your .env file.")
if not ENKRYPTAI_GUARDRAILS_NAME:
    raise ValueError("ENKRYPTAI_GUARDRAILS_NAME not found in environment variables. Please set it in your .env file.")

print("✅ Configuration loaded successfully")
print(f"📋 Guardrail Policy: {ENKRYPTAI_GUARDRAILS_NAME}")
print(f"📦 Batch Size: {BATCH_SIZE}")


## Step 2: Understanding Batch Processing

When processing large volumes of logs, we use **batch processing** to:
- **Improve efficiency**: Process multiple logs in a single API call
- **Reduce API overhead**: Fewer HTTP requests means faster processing
- **Respect rate limits**: Batch processing helps stay within API rate limits

The batch API endpoint accepts multiple texts at once and returns results for all of them.


## Step 3: Creating the Guardrails Function

This function handles the core logic of calling EnkryptAI Guardrails API with robust error handling:

**Key Features:**
- **Exponential Backoff**: Automatically retries on rate limit errors (429) with increasing delays
- **Error Handling**: Gracefully handles network errors and API failures
- **Timeout Protection**: Prevents hanging requests with a 30-second timeout


In [ ]:
def call_guardrails_batch(texts, guardrail_name, max_retries=5, initial_backoff=1):
    """
    Run multiple texts through EnkryptAI Guardrails to detect policy violations.
    
    This function implements exponential backoff retry logic to handle rate limiting
    gracefully. If the API returns a 429 (Too Many Requests) error, it will wait
    and retry with increasing delays: 1s, 2s, 4s, 8s, 16s.
    
    Args:
        texts (list): List of text strings to check for violations
        guardrail_name (str): The name of the guardrail policy to use
        max_retries (int): Maximum number of retry attempts for 429 errors (default: 5)
        initial_backoff (int): Initial backoff time in seconds (default: 1)
        
    Returns:
        dict: Guardrails API response containing violation detection results
    """
    # EnkryptAI Guardrails batch detection endpoint
    url = "https://api.enkryptai.com/guardrails/policy/batch/detect"
    
    # Set up request headers with authentication
    headers = {
        "Content-Type": "application/json",
        "apikey": ENKRYPTAI_API_KEY,
        "X-Enkrypt-Policy": guardrail_name  # Specify which guardrail policy to use
    }
    
    # Prepare the payload with all texts to check
    payload = {
        "texts": texts
    }
    
    # Initialize backoff time for exponential retry
    backoff_time = initial_backoff
    
    # Retry loop with exponential backoff
    for attempt in range(max_retries):
        try:
            # Make the API request
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            
            # Handle 429 Rate Limit errors with exponential backoff
            if response.status_code == 429:
                if attempt < max_retries - 1:
                    print(f"⚠️  Rate limit hit (429), retrying in {backoff_time}s... (attempt {attempt + 1}/{max_retries})")
                    time.sleep(backoff_time)
                    backoff_time *= 2  # Double the wait time for next retry
                    continue
                else:
                    return {"error": f"Rate limit exceeded after {max_retries} attempts"}
            
            # Raise an exception for other HTTP errors (4xx, 5xx)
            response.raise_for_status()
            
            # Return the successful response
            return response.json()
            
        except requests.exceptions.RequestException as e:
            # Handle network errors and other request exceptions
            if attempt < max_retries - 1 and "429" in str(e):
                print(f"⚠️  Rate limit detected, retrying in {backoff_time}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(backoff_time)
                backoff_time *= 2
                continue
            return {"error": str(e)}
        except Exception as e:
            # Handle any other unexpected errors
            return {"error": str(e)}
    
    return {"error": "Max retries exceeded"}

print("✅ Guardrails function defined")


## Step 4: Loading Your Logs

Now let's load your log file. The expected format is a JSON array where each log entry contains a `prompt` field:

```json
[
  {"prompt": "User input text here", ...other fields...},
  {"prompt": "Another user input", ...other fields...}
]
```


In [ ]:
# Load logs from file
try:
    with open(LOGS_FILE_PATH, 'r') as file:
        logs = json.load(file)
    
    # Extract prompts from logs
    # Assumes each log entry has a 'prompt' field
    prompts = [log['prompt'] for log in logs]
    total_prompts = len(prompts)
    
    print(f"✅ Successfully loaded {total_prompts} logs from {LOGS_FILE_PATH}")
    
    # Show a sample log entry for verification
    if logs:
        print(f"\n📝 Sample log entry structure:")
        print(json.dumps(logs[0], indent=2))
        print(f"\n📝 Sample prompt (first 100 chars):")
        print(f"   {prompts[0][:100]}...")
    
except FileNotFoundError:
    raise FileNotFoundError(f"Logs file not found: {LOGS_FILE_PATH}. Please ensure the file exists.")
except json.JSONDecodeError as e:
    raise ValueError(f"Invalid JSON in logs file: {e}")
except KeyError as e:
    raise KeyError(f"Logs file missing required field: {e}. Each log entry must have a 'prompt' field.")


## Step 5: Processing Logs in Batches

Now we'll process all logs through EnkryptAI Guardrails. The logs are divided into batches to:
- **Optimize API usage**: Process multiple logs per request
- **Handle large datasets**: Can process thousands of logs efficiently
- **Provide progress feedback**: See real-time progress as batches complete

**Processing Strategy:**
- Divide logs into batches of `BATCH_SIZE` (default: 10)
- Process each batch sequentially
- Add small delays between batches to avoid rate limits
- Collect all results for final analysis


In [ ]:
print(f"📊 Processing {total_prompts} logs in batches of {BATCH_SIZE}...")
print(f"📦 Estimated batches: {(total_prompts + BATCH_SIZE - 1) // BATCH_SIZE}\n")

# Initialize results storage
all_results = []
batch_count = 0
errors = []

# Process logs in batches
for i in range(0, total_prompts, BATCH_SIZE):
    batch_count += 1
    batch_end = min(i + BATCH_SIZE, total_prompts)
    batch_prompts = prompts[i:batch_end]
    
    print(f"🔄 Processing batch {batch_count} (logs {i+1}-{batch_end} of {total_prompts})...")
    
    # Call guardrails API for this batch
    batch_results = call_guardrails_batch(batch_prompts, ENKRYPTAI_GUARDRAILS_NAME)
    
    # Check if there was an error
    if "error" in batch_results:
        error_msg = batch_results['error']
        print(f"❌ Error in batch {batch_count}: {error_msg}")
        errors.append({
            "batch": batch_count,
            "range": f"{i+1}-{batch_end}",
            "error": error_msg
        })
        # Store error information with batch details
        all_results.append({
            "batch": batch_count,
            "range": f"{i+1}-{batch_end}",
            "error": error_msg
        })
    else:
        print(f"✅ Batch {batch_count} completed successfully")
        # Store results with batch information
        # Handle different response formats
        if isinstance(batch_results, dict) and 'results' in batch_results:
            # If API returns results in a 'results' field
            all_results.extend(batch_results['results'])
        elif isinstance(batch_results, list):
            # If API returns a list directly
            all_results.extend(batch_results)
        else:
            # Store the entire batch result
            all_results.append({
                "batch": batch_count,
                "range": f"{i+1}-{batch_end}",
                "results": batch_results
            })
    
    # Add a small delay between batches to avoid rate limiting
    # This helps prevent hitting rate limits even with successful requests
    if i + BATCH_SIZE < total_prompts:
        time.sleep(0.5)

print(f"\n✨ Processing complete! Processed {batch_count} batches.")
if errors:
    print(f"⚠️  Encountered {len(errors)} errors during processing.")
else:
    print("✅ All batches processed successfully!")


## Step 6: Analyzing Results

Let's examine the results to understand what violations were detected. This helps you:
- **Identify patterns**: See what types of violations are most common
- **Track compliance**: Understand your overall compliance rate
- **Investigate issues**: Find specific logs that need attention


In [ ]:
# Analyze results
print("📊 Results Summary:")
print(f"   Total logs processed: {total_prompts}")
print(f"   Total batches: {batch_count}")
print(f"   Total results: {len(all_results)}")
print(f"   Errors encountered: {len(errors)}")

# Count violations (assuming results have a 'policy_violation' or similar field)
# Adjust this based on your actual API response structure
violations_count = 0
if all_results:
    # Try to count violations - adjust field names based on your API response
    for result in all_results:
        if isinstance(result, dict):
            # Check common violation indicators
            if result.get('policy_violation') == 1 or result.get('violation') == True:
                violations_count += 1
            elif isinstance(result.get('summary'), dict) and result.get('summary', {}).get('policy_violation') == 1:
                violations_count += 1

if violations_count > 0:
    print(f"   ⚠️  Violations detected: {violations_count}")
    print(f"   ✅ Clean logs: {total_prompts - violations_count}")
else:
    print(f"   ✅ No violations detected in processed logs")

# Show a sample result if available
if all_results and not errors:
    print(f"\n📝 Sample result structure:")
    sample_result = all_results[0] if all_results else None
    if sample_result:
        print(json.dumps(sample_result, indent=2))


## Step 7: Saving Results

Finally, we'll save all results to a JSON file for further analysis, reporting, or integration with other systems.

The output file includes:
- **Metadata**: Total logs, batch size, batch count
- **Results**: All guardrails detection results
- **Errors**: Any errors encountered during processing


In [ ]:
# Prepare output data with metadata and results
output_data = {
    "metadata": {
        "total_logs": total_prompts,
        "batch_size": BATCH_SIZE,
        "total_batches": batch_count,
        "guardrail_policy": ENKRYPTAI_GUARDRAILS_NAME,
        "processing_timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    },
    "summary": {
        "total_results": len(all_results),
        "errors": len(errors),
        "violations_detected": violations_count if 'violations_count' in locals() else "N/A"
    },
    "results": all_results
}

# Save results to file
output_filename = 'guardrails_results.json'
with open(output_filename, 'w') as file:
    json.dump(output_data, file, indent=4)

print(f"💾 Results saved to {output_filename}")
print(f"📁 File contains:")
print(f"   - Metadata: processing information and configuration")
print(f"   - Summary: high-level statistics")
print(f"   - Results: detailed guardrails detection results for each log")


## Next Steps

Now that you have your results, you can:

1. **Review Violations**: Examine logs that triggered policy violations
2. **Generate Reports**: Create compliance reports from the results
3. **Set Up Monitoring**: Integrate this process into your regular audit workflow
4. **Investigate Patterns**: Look for common violation types or sources

## Tips for Production Use

- **Adjust Batch Size**: Increase `BATCH_SIZE` for faster processing, decrease if hitting rate limits
- **Schedule Regular Runs**: Process logs daily/weekly to maintain compliance
- **Monitor Errors**: Set up alerts for API errors or rate limit issues
- **Archive Results**: Keep historical results for trend analysis

## Need Help?

- Check the [EnkryptAI Documentation](https://docs.enkryptai.com) for API details
- Review your guardrail policy configuration in the EnkryptAI dashboard
- Contact support if you encounter persistent issues
